# Interactive CARMENS RFI Scaling Notebook

This notebook starts from turboSETI detections in the .dat files, resolves the matching HDF5 data, and plots waterfall data at progressively wider frequency spans to show whether a narrowband feature sits inside a broader RFI structure.


In [ ]:
%matplotlib inline

import os
import re
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import h5py
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display

from blimpy import Waterfall

# Project paths
ROOT = Path('/home/wlll2x/turboSETI')
CSV_PATH = ROOT / 'scripts' / 'carmenes_1688mhz_subset.csv'
OUTPUT_DIR = ROOT / 'notebooks' / 'outputs' / 'carmen_rfi_interactive'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Zoom ladder tuned for the kHz-to-MHz transition where RFI morphology becomes informative.
SPAN_OPTIONS_MHZ = [0.0002, 0.002, 0.02, 0.2, 2.0, 20.0, 100.0]
DEFAULT_SPAN_MHZ = 0.2
SPAN_LABELS = {
    0.0002: 'sub-kHz',
    0.002: 'few-kHz',
    0.02: '10-20 kHz',
    0.2: 'RFI structure window',
    2.0: 'broad RFI',
    20.0: 'bandpass',
    100.0: 'full-band context',
}


def load_csv_candidates(csv_path: Path, targets: List[str]) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df = df[df['Target'].astype(str).str.strip().isin(targets)].copy()
    df['h5_path'] = df['.h5 path'].astype(str).str.strip()
    df['dat_path'] = df['.dat path'].astype(str).str.strip()
    df['Frequency'] = pd.to_numeric(df['Frequency'], errors='coerce')
    return df.reset_index(drop=True)


def resolve_h5_path(row: pd.Series) -> Optional[Path]:
    path = row.get('h5_path', '')
    if not isinstance(path, str):
        return None
    candidate = Path(path)
    if candidate.exists():
        return candidate
    alt = Path('/datax/scratch/wlll2x/raw') / candidate.name
    if alt.exists():
        return alt
    return None


def resolve_dat_path(row: pd.Series) -> Optional[Path]:
    path = row.get('dat_path', '')
    if not isinstance(path, str):
        return None
    candidate = Path(path)
    if candidate.exists():
        return candidate
    alt = Path('/datax/scratch/wlll2x/raw') / candidate.name
    if alt.exists():
        return alt
    return None


def read_turboseti_detections(dat_path: Optional[Path]) -> pd.DataFrame:
    if dat_path is None or not Path(dat_path).exists():
        return pd.DataFrame()
    try:
        det_df = pd.read_csv(dat_path, comment='#', sep=r'\s+', engine='python')
    except Exception as exc:
        print(f'Could not parse detection file {dat_path}: {exc}')
        return pd.DataFrame()

    if det_df.empty:
        return pd.DataFrame()

    freq_cols = [c for c in ('Corrected_Frequency', 'Uncorrected_Frequency', 'Frequency') if c in det_df.columns]
    if not freq_cols:
        return pd.DataFrame()

    for col in freq_cols:
        det_df[col] = pd.to_numeric(det_df[col], errors='coerce')

    det_df['frequency_mhz'] = det_df[freq_cols[0]]
    for col in freq_cols[1:]:
        det_df['frequency_mhz'] = det_df['frequency_mhz'].fillna(det_df[col])

    det_df = det_df.dropna(subset=['frequency_mhz']).copy()
    if det_df.empty:
        return pd.DataFrame()

    drift_cols = [c for c in ('Drift_Rate', 'DriftRate', 'drift_rate') if c in det_df.columns]
    if drift_cols:
        det_df['drift_rate_hz_s'] = pd.to_numeric(det_df[drift_cols[0]], errors='coerce')
    else:
        det_df['drift_rate_hz_s'] = np.nan

    if 'SNR' in det_df.columns:
        det_df['snr'] = pd.to_numeric(det_df['SNR'], errors='coerce')
    else:
        det_df['snr'] = np.nan

    return det_df[['frequency_mhz', 'snr', 'drift_rate_hz_s'] + [c for c in det_df.columns if c not in ['frequency_mhz', 'snr', 'drift_rate_hz_s']]]


def build_detection_candidates(csv_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in csv_df.iterrows():
        resolved_h5 = resolve_h5_path(row)
        resolved_dat = resolve_dat_path(row)
        detections = read_turboseti_detections(resolved_dat)

        if detections.empty:
            if pd.notna(row.get('Frequency')):
                records.append({
                    'Target': row.get('Target', ''),
                    'Session': row.get('Session', ''),
                    'Cadence ID': row.get('Cadence ID', ''),
                    'frequency_mhz': float(row['Frequency']),
                    'h5_path': str(resolved_h5) if resolved_h5 is not None else '',
                    'dat_path': str(resolved_dat) if resolved_dat is not None else '',
                    'drift_rate_hz_s': np.nan,
                    'source': 'csv_fallback',
                })
            continue

        for idx, det in detections.iterrows():
            drift_rate = det.get('drift_rate_hz_s', np.nan)
            records.append({
                'Target': row.get('Target', ''),
                'Session': row.get('Session', ''),
                'Cadence ID': row.get('Cadence ID', ''),
                'frequency_mhz': float(det['frequency_mhz']),
                'h5_path': str(resolved_h5) if resolved_h5 is not None else '',
                'dat_path': str(resolved_dat) if resolved_dat is not None else '',
                'drift_rate_hz_s': float(drift_rate) if pd.notna(drift_rate) else np.nan,
                'source': 'turboseti_dat',
                'detection_index': idx,
            })

    if not records:
        return pd.DataFrame()

    return pd.DataFrame(records).reset_index(drop=True)


def get_waterfall_arrays(wf) -> Tuple[np.ndarray, np.ndarray, float]:
    data = wf.data
    if data.ndim == 3 and data.shape[1] >= 1:
        data = data[:, 0, :]
    if data.ndim != 2:
        raise ValueError(f'Unexpected waterfall shape: {data.shape}')

    fch1 = float(wf.header.get('fch1', 0.0))
    foff = float(wf.header.get('foff', 1.0))
    nchans = int(wf.header.get('nchans', data.shape[1]))
    freqs = fch1 + np.arange(nchans, dtype=float) * foff

    if freqs.size != data.shape[1]:
        freqs = np.resize(freqs, data.shape[1])

    if freqs[0] > freqs[-1]:
        data = data[:, ::-1]
        freqs = freqs[::-1]

    tsamp = float(wf.header.get('tsamp', 1.0))
    return data, freqs, tsamp


@lru_cache(maxsize=16)
def load_waterfall(path: str) -> Dict[str, Any]:
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f'Waterfall file not found: {path_obj}')

    wf = Waterfall(str(path_obj), load_data=True)
    data, freqs, tsamp = get_waterfall_arrays(wf)
    data = np.asarray(data, dtype=float)

    if data.ndim != 2:
        raise ValueError(f'Unexpected data shape for plotting: {data.shape}')

    log_data = np.log10(data + 1e-12)
    vmin = float(np.percentile(log_data, 2))
    vmax = float(np.percentile(log_data, 99.5))
    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin + 1e-3

    return {
        'path': str(path_obj),
        'data': data,
        'freqs': freqs,
        'tsamp': tsamp,
        'vmin': float(vmin),
        'vmax': float(vmax),
    }


def slice_waterfall(wf_cache: Dict[str, Any], center_freq: float, span_mhz: float, downsample: bool = True) -> Tuple[np.ndarray, np.ndarray, Tuple[float, float, float, float], float]:
    data = wf_cache['data']
    freqs = wf_cache['freqs']
    tsamp = wf_cache['tsamp']
    vmin = wf_cache['vmin']
    vmax = wf_cache['vmax']

    f_start = center_freq - span_mhz / 2.0
    f_stop = center_freq + span_mhz / 2.0
    freq_mask = (freqs >= f_start) & (freqs <= f_stop)

    if not np.any(freq_mask):
        raise ValueError(f'No frequency channels selected for {f_start:.6f} to {f_stop:.6f} MHz')

    data = data[:, freq_mask]
    freqs = freqs[freq_mask]

    time_step = 1
    freq_step = 1
    if downsample:
        time_step = max(1, data.shape[0] // 120)
        freq_step = max(1, data.shape[1] // 2048)

    data = data[::time_step, ::freq_step]
    freqs = freqs[::freq_step]

    log_data = np.log10(data + 1e-12)
    data_norm = (log_data - vmin) / (vmax - vmin + 1e-12)
    data_norm = np.clip(data_norm, 0, 1)

    duration_s = data.shape[0] * tsamp * time_step
    extent = (float(freqs[0]), float(freqs[-1]), 0.0, float(duration_s))
    return data_norm, freqs, extent, duration_s


def build_plot_title(record: Optional[Dict[str, Any]], center_freq: float, span_mhz: float, drift_rate_hz_s: Optional[float] = None) -> str:
    target = record.get('Target', 'candidate') if record is not None else 'candidate'
    span_label = SPAN_LABELS.get(span_mhz, 'custom')
    title_bits = [f'{target}', f'center {center_freq:.6f} MHz', f'span {span_mhz:.3f} MHz ({span_label})']
    if drift_rate_hz_s is not None and np.isfinite(drift_rate_hz_s):
        title_bits.append(f'drift {drift_rate_hz_s:.2f} Hz/s')
    return ' | '.join(title_bits)


def render_waterfall_to_axes(ax, waterfall_obj, center_freq: float, span_mhz: float, drift_rate_hz_s: Optional[float] = None, title: Optional[str] = None, record: Optional[Dict[str, Any]] = None):
    if isinstance(waterfall_obj, (str, Path)):
        wf_cache = load_waterfall(str(waterfall_obj))
    else:
        wf_cache = waterfall_obj

    data_norm, freqs, extent, duration_s = slice_waterfall(wf_cache, center_freq, span_mhz)

    ax.clear()
    image = ax.imshow(data_norm, aspect='auto', origin='lower', extent=extent, cmap='viridis')

    fig = ax.figure
    fig.colorbar(image, ax=ax, label='log power (normalized)')
    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('Time [s]')
    ax.set_title(title or build_plot_title(record, center_freq, span_mhz, drift_rate_hz_s))

    if drift_rate_hz_s is not None and np.isfinite(drift_rate_hz_s) and span_mhz <= 0.2:
        times = np.linspace(0.0, float(duration_s), 200)
        line_freqs = center_freq + (times * drift_rate_hz_s / 1e6)
        ax.plot(line_freqs, times, color='red', linewidth=2.2, alpha=0.95, zorder=10)
        ax.axvline(center_freq, color='white', linewidth=1.2, alpha=0.8, zorder=10)

    fig.tight_layout()
    return image


def plot_waterfall(waterfall_obj, center_freq: float, span_mhz: float, title: Optional[str] = None,
                   show: bool = True, return_fig: bool = False, drift_rate_hz_s: Optional[float] = None,
                   record: Optional[Dict[str, Any]] = None):
    fig, ax = plt.subplots(figsize=(14, 5))
    render_waterfall_to_axes(ax, waterfall_obj, center_freq=center_freq, span_mhz=span_mhz,
                             drift_rate_hz_s=drift_rate_hz_s, title=title, record=record)
    if show:
        display(fig)
    if return_fig:
        return fig
    plt.close(fig)
    return None


def plot_candidate(waterfall_obj, center_freq: float, span_mhz: float, show: bool = True,
                   drift_rate_hz_s: Optional[float] = None, record: Optional[Dict[str, Any]] = None):
    return plot_waterfall(waterfall_obj, center_freq=center_freq, span_mhz=span_mhz,
                          title=f'Waterfall around {center_freq:.4f} MHz',
                          show=show, return_fig=True, drift_rate_hz_s=drift_rate_hz_s, record=record)


def save_detection_plot(waterfall_obj, record: Optional[Dict[str, Any]], center_freq: float, span_mhz: float,
                         prefix: str, drift_rate_hz_s: Optional[float] = None) -> Path:
    fig = plot_waterfall(waterfall_obj, center_freq=center_freq, span_mhz=span_mhz, show=False,
                         return_fig=True, drift_rate_hz_s=drift_rate_hz_s,
                         title=build_plot_title(record, center_freq, span_mhz, drift_rate_hz_s),
                         record=record)
    target = str(record.get('Target', 'candidate')).strip() if record is not None else 'candidate'
    target = re.sub(r'[^A-Za-z0-9._-]+', '_', target)
    stem = f'{target}_{prefix}_cf{center_freq:.6f}_span{span_mhz:.3f}MHz'
    out_path = OUTPUT_DIR / f'{stem}.png'
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return out_path


def create_comparison_figure(waterfall_obj, record: Optional[Dict[str, Any]], center_freq: float,
                             spans: Optional[List[float]] = None, drift_rate_hz_s: Optional[float] = None) -> plt.Figure:
    spans = spans or [0.0002, 0.02, 0.2, 2.0]
    fig, axes = plt.subplots(1, len(spans), figsize=(4 * len(spans), 4.5), sharey=True)
    if len(spans) == 1:
        axes = [axes]
    else:
        axes = list(axes)

    for ax, span in zip(axes, spans):
        render_waterfall_to_axes(
            ax,
            waterfall_obj,
            center_freq=center_freq,
            span_mhz=float(span),
            drift_rate_hz_s=drift_rate_hz_s,
            title=build_plot_title(record, center_freq, float(span), drift_rate_hz_s),
            record=record,
        )

    fig.suptitle(f"{record.get('Target', 'candidate')} | center {center_freq:.6f} MHz | multi-scale morphology", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


print('Loaded imports and helper functions.')


In [ ]:
# Define targets to analyze
targets = ['NGC5638', 'SAG_DIR']
print(f"Will search for targets: {targets}")


In [ ]:
df = load_csv_candidates(CSV_PATH, targets)

df["Target"] = df["Target"].astype(str).str.strip()
df["Frequency"] = pd.to_numeric(df["Frequency"], errors="coerce")

existing_targets = sorted(df["Target"].unique())

missing_targets = [t for t in targets if t not in existing_targets]

print("Requested targets:", targets)
print("Found targets in CSV:", existing_targets)

if missing_targets:
    print("Missing from CSV:", missing_targets)

print("Rows matched:", len(df))


In [ ]:
detection_candidates = build_detection_candidates(df)
if detection_candidates.empty:
    raise RuntimeError('No turboSETI detections or CSV fallback frequencies were found.')

detection_candidates[["Target", "Session", "Cadence ID", "frequency_mhz", "h5_path", "dat_path", "source"]].head(10)


In [ ]:
# Start from the first turboSETI detection and use it as the center frequency.
detection = detection_candidates.iloc[0]
resolved_path = Path(detection["h5_path"]) if detection.get("h5_path") else None
resolved_dat_path = Path(detection["dat_path"]) if detection.get("dat_path") else None
center_freq = float(detection["frequency_mhz"])
drift_rate_hz_s = float(detection.get("drift_rate_hz_s", np.nan)) if pd.notna(detection.get("drift_rate_hz_s", np.nan)) else None
print("Selected detection:")
print(detection.to_dict())
print("Resolved HDF5 path:", resolved_path)
print("Resolved detection path:", resolved_dat_path)
print("Center frequency for plotting:", center_freq)
print("Drift rate [Hz/s]:", drift_rate_hz_s)


In [ ]:
plot_waterfall(resolved_path, center_freq=center_freq, span_mhz=DEFAULT_SPAN_MHZ, show=True, drift_rate_hz_s=drift_rate_hz_s, record=detection)


In [ ]:
span_slider = widgets.SelectionSlider(
    options=SPAN_OPTIONS_MHZ,
    value=DEFAULT_SPAN_MHZ,
    description='span [MHz]',
    continuous_update=False,
)

detection_dropdown = widgets.Dropdown(
    options=[(f"{row['Target']} @ {row['frequency_mhz']:.6f} MHz", idx) for idx, row in detection_candidates.iterrows()],
    value=0,
    description='detection',
)

out = widgets.Output()


def update_plot(selection_idx, span_mhz):
    with out:
        clear_output(wait=True)
        record = detection_candidates.iloc[selection_idx]
        resolved_path = Path(record['h5_path']) if record.get('h5_path') else None
        center_freq = float(record['frequency_mhz'])
        drift_rate_hz_s = float(record.get('drift_rate_hz_s', np.nan)) if pd.notna(record.get('drift_rate_hz_s', np.nan)) else None
        fig, ax = plt.subplots(figsize=(14, 5))
        render_waterfall_to_axes(
            ax,
            resolved_path,
            center_freq=center_freq,
            span_mhz=float(span_mhz),
            drift_rate_hz_s=drift_rate_hz_s,
            title=build_plot_title(record, center_freq, float(span_mhz), drift_rate_hz_s),
            record=record,
        )
        display(fig)
        plt.close(fig)

widgets.interactive_output(update_plot, {'selection_idx': detection_dropdown, 'span_mhz': span_slider})
display(widgets.HBox([detection_dropdown, span_slider]))
display(out)


In [ ]:
for span in SPAN_OPTIONS_MHZ:
    fig = plot_waterfall(
        resolved_path,
        center_freq=center_freq,
        span_mhz=span,
        show=False,
        return_fig=True,
        drift_rate_hz_s=drift_rate_hz_s,
        record=detection,
    )
    plt.close(fig)


In [ ]:
saved_paths = []
for span, prefix in [(0.00002, 'narrow'), (0.2, 'midres')]:
    out_path = save_detection_plot(
        resolved_path,
        detection,
        center_freq=center_freq,
        span_mhz=span,
        prefix=prefix,
        drift_rate_hz_s=drift_rate_hz_s,
    )
    saved_paths.append(out_path)

print('Saved plots:')
for path in saved_paths:
    print(path)


In [ ]:
for span in SPAN_OPTIONS_MHZ:
    fig = plot_candidate(
        resolved_path,
        center_freq=center_freq,
        span_mhz=span,
        show=False,
        drift_rate_hz_s=drift_rate_hz_s,
        record=detection,
    )
    plt.close(fig)


In [ ]:
saved_paths = []
for span, prefix in [(0.00002, 'narrow'), (0.2, 'midres')]:
    out_path = save_detection_plot(
        resolved_path,
        detection,
        center_freq=center_freq,
        span_mhz=span,
        prefix=prefix,
        drift_rate_hz_s=drift_rate_hz_s,
    )
    saved_paths.append(out_path)

print('Saved plots:')
for path in saved_paths:
    print(path)
